In [1]:
import ast
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

base_folder = f'/active-data/analysis_results/chr_pla/genus'
figure_data = f'{base_folder}/figure_data'
os.makedirs(figure_data, exist_ok=True)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
# kmer_data
def load_single_kmer(kmer_dir, record_id, target_k):
    kmer_file = os.path.join(kmer_dir, f"{record_id}.txt")
    if not os.path.exists(kmer_file):
        return None
    with open(kmer_file, 'r') as f:
        raw = f.read()
        kmer_dict = ast.literal_eval(raw)
        
    k_key = f"{target_k}-mer"
    if k_key not in kmer_dict:
        return None
    kmer_counts = kmer_dict[k_key]
    valid_bases = {"A", "T", "C", "G"}
    clean_counts = {}
    for seq, cnt in kmer_counts.items():
        if set(seq).issubset(valid_bases):
            clean_counts[seq] = cnt
    total = sum(clean_counts.values())
    if total == 0:
        return None
    return {f"{seq}": cnt / total for seq, cnt in clean_counts.items()}

def analyze_group_kmer(genus_name):
    kmer_base_dir = "/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/kmer"
    k_values = [3, 4, 5]
    label = "pident_90"
    group_col = f"category-{label}"
    sample_per_group = 200
    repeat_times = 5
    
    folder = os.path.join(base_folder, "statistics_records", genus_name)
    csv_path = os.path.join(folder, "replicon-plasmid_fraction-self_bitscore_statistics.csv")
    
    out_folder = os.path.join(figure_data, "kmer_analysis_output", genus_name)
    os.makedirs(out_folder, exist_ok=True)
    
    replicon_df = pd.read_csv(csv_path)
    df = replicon_df.dropna(subset=[group_col]).copy()
    all_groups = sorted(df[group_col].unique())

    chr_frag_pd = pd.read_csv(f'{base_folder}/kmer_chr_frag_samples/{genus_name}/chromosome_fragment_data.tsv', sep='\t')
    chr_frag_label = 'chromosome fragment'
    all_groups.append(chr_frag_label)

    pairwise_res = []
    variance_summary = []

    for k in k_values:
        sample_list = []
        group_list = []
        acc_list = []
        
        for idx, row in df.iterrows():
            acc_n = row["accession"].split('-')[0]
            record_id = row["accession"].split('-')[1]
            full_acc = row["accession"]
            g = row[group_col]
            kmer_dir = os.path.join(kmer_base_dir, acc_n)
            kmer_freq = load_single_kmer(kmer_dir, record_id, k)
            if kmer_freq is None:
                continue
            sample_list.append(kmer_freq)
            group_list.append(g)
            acc_list.append(full_acc)

        for idx, row in chr_frag_pd.iterrows():
            full_acc = row["sequence"]
            g = chr_frag_label
            kmer_dir = f'{base_folder}/kmer_chr_frag_samples/{genus_name}/fragment_records'
            kmer_freq = load_single_kmer(kmer_dir, full_acc, k)
            sample_list.append(kmer_freq)
            group_list.append(g)
            acc_list.append(full_acc)            

        for repeat_idx in range(1, repeat_times + 1):
            group_dict = defaultdict(list)
            for s, g, acc in zip(sample_list, group_list, acc_list):
                group_dict[g].append((s, acc))
            
            sample_sam = []
            group_sam = []
            acc_sam = []
            for g, lst in group_dict.items():
                if len(lst) > sample_per_group:
                    lst = random.sample(lst, sample_per_group)
                
                selected_samples = [item[0] for item in lst]
                selected_accs = [item[1] for item in lst]
                
                sample_sam.extend(selected_samples)
                group_sam.extend([g] * len(selected_samples))
                acc_sam.extend(selected_accs)

            all_kmers = set()
            for s in sample_sam:
                all_kmers.update(s.keys())
            all_kmers = sorted(all_kmers)

            # ANOSIM
            for g1, g2 in combinations(all_groups, 2):
                sub_idx = [i for i, g in enumerate(group_sam) if g in (g1, g2)]
                sub_samples = [sample_sam[i] for i in sub_idx]
                sub_groups = [group_sam[i] for i in sub_idx]
                
                mat = np.array([[s.get(kmer, 0.0) for kmer in all_kmers] for s in sub_samples])
                lab = [0 if g == g1 else 1 for g in sub_groups]
                
                dist_mat = pdist(mat, metric="braycurtis")
                dm = DistanceMatrix(dist_mat)
                res = anosim(dm, lab, permutations=999)
                
                r = res['test statistic']
                p = res['p-value']
                
                pairwise_res.append({
                    "genus": genus_name, "k": k, "repeat": repeat_idx,
                    "group1": g1, "group2": g2,
                    "R": r, "P": p,
                    "significant": p < 0.05
                })

            # PCA
            mat_all = np.array([[s.get(kmer, 0.0) for kmer in all_kmers] for s in sample_sam])
            pca = PCA(n_components=2)
            coord = pca.fit_transform(mat_all)
            
            var_pc1 = pca.explained_variance_ratio_[0] * 100
            var_pc2 = pca.explained_variance_ratio_[1] * 100

            pca_df = pd.DataFrame({
                "accession": acc_sam, 
                "group": group_sam,
                "PC1": coord[:, 0], 
                "PC2": coord[:, 1],
                "PC1_variance_pct": var_pc1,
                "PC2_variance_pct": var_pc2,
                "k": k, 
                "genus": genus_name, 
                "repeat": repeat_idx
            })
            pca_out_path = os.path.join(out_folder, f"{k}mer_pca_coords_repeat{repeat_idx}.csv")
            pca_df.to_csv(pca_out_path, index=False)

            variance_summary.append({
                "genus": genus_name,
                "kmer": k,
                "repeat": repeat_idx,
                "PC1_variance_pct": var_pc1,
                "PC2_variance_pct": var_pc2,
                "total_variance_pct": var_pc1 + var_pc2
            })

    variance_df = pd.DataFrame(variance_summary)
    variance_out_path = os.path.join(out_folder, "pca_variance_5repeats_summary.csv")
    variance_df.to_csv(variance_out_path, index=False)

    anosim_df = pd.DataFrame(pairwise_res)
    anosim_out_path = os.path.join(out_folder, "anosim_pairwise_results_5repeats.csv")
    anosim_df.to_csv(anosim_out_path, index=False)

In [3]:
from collections import defaultdict
from itertools import combinations
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from skbio.stats.distance import anosim, DistanceMatrix
import numpy as np
import random

for genus_name in keep_genus:
    print(genus_name)
    %time analyze_group_kmer(genus_name)

Escherichia
CPU times: user 5min 43s, sys: 5.84 s, total: 5min 49s
Wall time: 5min 27s
Klebsiella
CPU times: user 5min 38s, sys: 4.96 s, total: 5min 42s
Wall time: 5min 20s
Staphylococcus
CPU times: user 2min 53s, sys: 2.11 s, total: 2min 55s
Wall time: 2min 41s
Pseudomonas
CPU times: user 2min 24s, sys: 1.37 s, total: 2min 25s
Wall time: 2min 10s
Bacillus
CPU times: user 2min 47s, sys: 1.75 s, total: 2min 49s
Wall time: 2min 32s
Salmonella
CPU times: user 2min 50s, sys: 1.91 s, total: 2min 52s
Wall time: 2min 37s
Streptococcus
CPU times: user 1min 47s, sys: 852 ms, total: 1min 48s
Wall time: 1min 34s
Streptomyces
CPU times: user 2min 15s, sys: 1.22 s, total: 2min 16s
Wall time: 2min
Acinetobacter
CPU times: user 2min 32s, sys: 1.56 s, total: 2min 34s
Wall time: 2min 18s
Enterococcus
CPU times: user 2min 22s, sys: 1.42 s, total: 2min 23s
Wall time: 2min 8s
Bordetella
CPU times: user 2min 38s, sys: 484 ms, total: 2min 38s
Wall time: 58.2 s
Enterobacter
CPU times: user 2min 13s, sys: 1 s